# Notebook 6: Train, Tune & Evaluate
**MLOps Training 2026/2027 · Task 2: From Tables to Notebooks**

### Objective:
1. Load the engineered feature datasets from Notebook 5.
2. Establish a **dummy baseline** (predict majority class) to quantify the minimum performance bar.
3. Train a **Logistic Regression** (linear baseline) and a **Random Forest** classifier.
4. Tune the best model using `RandomizedSearchCV` on the validation set.
5. Perform **final evaluation on the held-out test set** (used exactly once).
6. Save the best trained model and evaluation metrics as artifacts.

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from dotenv import load_dotenv
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
def find_project_root(start_path: Path) -> Path:
    for parent in [start_path] + list(start_path.parents):
        if (parent / '.env').exists() or (parent / '.git').exists():
            return parent
    return start_path

root_dir = find_project_root(Path.cwd())
load_dotenv(dotenv_path=root_dir / '.env')

fe_dir = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '05_feature_engineering'
output_dir = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '06_train_tune_evaluate'
output_dir.mkdir(parents=True, exist_ok=True)

X_train = pd.read_parquet(fe_dir / 'X_train.parquet').values
X_val   = pd.read_parquet(fe_dir / 'X_val.parquet').values
X_test  = pd.read_parquet(fe_dir / 'X_test.parquet').values
y_train = pd.read_parquet(fe_dir / 'y_train.parquet')['is_late'].values
y_val   = pd.read_parquet(fe_dir / 'y_val.parquet')['is_late'].values
y_test  = pd.read_parquet(fe_dir / 'y_test.parquet')['is_late'].values

with open(fe_dir / 'feature_names.txt') as f:
    feature_names = f.read().splitlines()

print(f'X_train: {X_train.shape}, y_train late rate: {y_train.mean()*100:.2f}%')
print(f'X_val  : {X_val.shape},   y_val late rate  : {y_val.mean()*100:.2f}%')
print(f'X_test : {X_test.shape},  y_test late rate : {y_test.mean()*100:.2f}%')
print(f'Total features: {len(feature_names)}')

X_train: (67529, 145), y_train late rate: 8.11%
X_val  : (14470, 145),   y_val late rate  : 8.11%
X_test : (14471, 145),  y_test late rate : 8.11%
Total features: 145


## Step 1: Dummy Baseline

A `DummyClassifier` that always predicts the majority class (0 = On-Time) sets the minimum bar.
Any trained model **must beat this baseline** to be considered useful.

In [3]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)

y_val_pred_dummy = dummy.predict(X_val)
dummy_f1   = f1_score(y_val, y_val_pred_dummy, pos_label=1, zero_division=0)
dummy_roc  = roc_auc_score(y_val, dummy.predict_proba(X_val)[:, 1])
dummy_pr   = average_precision_score(y_val, dummy.predict_proba(X_val)[:, 1])

print('=== Dummy Baseline (Always Predict On-Time) ===')
print(f'  F1 (Late class)   : {dummy_f1:.4f}')
print(f'  ROC-AUC           : {dummy_roc:.4f}')
print(f'  PR-AUC            : {dummy_pr:.4f}')
print('\nClassification Report:')
print(classification_report(y_val, y_val_pred_dummy, target_names=['On-Time','Late'], zero_division=0))

=== Dummy Baseline (Always Predict On-Time) ===
  F1 (Late class)   : 0.0000
  ROC-AUC           : 0.5000
  PR-AUC            : 0.0811

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.92      1.00      0.96     13296
        Late       0.00      0.00      0.00      1174

    accuracy                           0.92     14470
   macro avg       0.46      0.50      0.48     14470
weighted avg       0.84      0.92      0.88     14470



## Step 2: Logistic Regression (Linear Baseline)

In [4]:
lr = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)

y_val_prob_lr = lr.predict_proba(X_val)[:, 1]
y_val_pred_lr = lr.predict(X_val)

lr_f1  = f1_score(y_val, y_val_pred_lr, pos_label=1)
lr_roc = roc_auc_score(y_val, y_val_prob_lr)
lr_pr  = average_precision_score(y_val, y_val_prob_lr)

print('=== Logistic Regression (Val Set) ===')
print(f'  F1 (Late class)   : {lr_f1:.4f}')
print(f'  ROC-AUC           : {lr_roc:.4f}')
print(f'  PR-AUC            : {lr_pr:.4f}')
print('\nClassification Report:')
print(classification_report(y_val, y_val_pred_lr, target_names=['On-Time','Late']))

=== Logistic Regression (Val Set) ===
  F1 (Late class)   : 0.2800
  ROC-AUC           : 0.7788
  PR-AUC            : 0.3393

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.96      0.72      0.82     13296
        Late       0.18      0.68      0.28      1174

    accuracy                           0.72     14470
   macro avg       0.57      0.70      0.55     14470
weighted avg       0.90      0.72      0.78     14470



## Step 3: Random Forest with Hyperparameter Tuning

We use `RandomizedSearchCV` with ROC-AUC scoring on a 3-fold cross-validation to tune the Random Forest. Tuning is done on the **training set only**.

In [5]:
param_dist = {
    'n_estimators':      [100, 150],
    'max_depth':         [6, 10, 15],
    'min_samples_split': [5, 10],
    'min_samples_leaf':  [2, 4],
    'class_weight':      ['balanced']
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=6,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_train, y_train)

print(f'Best params : {search.best_params_}')
print(f'Best CV ROC-AUC: {search.best_score_:.4f}')

Fitting 3 folds for each of 6 candidates, totalling 18 fits


Best params : {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 15, 'class_weight': 'balanced'}
Best CV ROC-AUC: 0.8101


In [6]:
best_rf = search.best_estimator_

y_val_prob_rf = best_rf.predict_proba(X_val)[:, 1]
y_val_pred_rf = best_rf.predict(X_val)

rf_f1  = f1_score(y_val, y_val_pred_rf, pos_label=1)
rf_roc = roc_auc_score(y_val, y_val_prob_rf)
rf_pr  = average_precision_score(y_val, y_val_prob_rf)

print('=== Tuned Random Forest (Val Set) ===')
print(f'  F1 (Late class)   : {rf_f1:.4f}')
print(f'  ROC-AUC           : {rf_roc:.4f}')
print(f'  PR-AUC            : {rf_pr:.4f}')
print('\nClassification Report:')
print(classification_report(y_val, y_val_pred_rf, target_names=['On-Time','Late']))

=== Tuned Random Forest (Val Set) ===
  F1 (Late class)   : 0.3977
  ROC-AUC           : 0.8130
  PR-AUC            : 0.3617

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.96      0.90      0.93     13296
        Late       0.32      0.52      0.40      1174

    accuracy                           0.87     14470
   macro avg       0.64      0.71      0.66     14470
weighted avg       0.90      0.87      0.89     14470



## Step 4: Model Comparison (Validation Set)

In [7]:
comparison = pd.DataFrame([
    {'Model': 'Dummy (Majority Class)',          'F1 (Late)': dummy_f1,  'ROC-AUC': dummy_roc,  'PR-AUC': dummy_pr},
    {'Model': 'Logistic Regression (balanced)',  'F1 (Late)': lr_f1,    'ROC-AUC': lr_roc,    'PR-AUC': lr_pr},
    {'Model': 'Tuned Random Forest (balanced)', 'F1 (Late)': rf_f1,    'ROC-AUC': rf_roc,    'PR-AUC': rf_pr},
]).set_index('Model').round(4)

print('=== Validation Set Model Comparison ===')
display(comparison)

=== Validation Set Model Comparison ===


,F1 (Late),ROC-AUC,PR-AUC
Model,,,
Dummy (Majority Class),0.0000,0.5000,0.0811
Logistic Regression (balanced),0.2800,0.7788,0.3393
Tuned Random Forest (balanced),0.3977,0.8130,0.3617


## Step 5: Final Test Set Evaluation

> **The test set is used exactly once** — here, at the very end. We evaluate the best model (Tuned Random Forest) on the held-out test set to get an unbiased estimate of generalization performance.

In [8]:
# Final evaluation on test set
y_test_prob = best_rf.predict_proba(X_test)[:, 1]
y_test_pred = best_rf.predict(X_test)

test_f1  = f1_score(y_test, y_test_pred, pos_label=1)
test_roc = roc_auc_score(y_test, y_test_prob)
test_pr  = average_precision_score(y_test, y_test_prob)

print('=== FINAL TEST SET EVALUATION (Tuned Random Forest) ===')
print(f'  F1 (Late class)   : {test_f1:.4f}')
print(f'  ROC-AUC           : {test_roc:.4f}')
print(f'  PR-AUC            : {test_pr:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_test_pred, target_names=['On-Time','Late']))

=== FINAL TEST SET EVALUATION (Tuned Random Forest) ===
  F1 (Late class)   : 0.3913
  ROC-AUC           : 0.8156
  PR-AUC            : 0.3528

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.96      0.90      0.93     13297
        Late       0.31      0.52      0.39      1174

    accuracy                           0.87     14471
   macro avg       0.63      0.71      0.66     14471
weighted avg       0.90      0.87      0.88     14471



## Step 6: Feature Importance

In [9]:
importances = pd.Series(best_rf.feature_importances_, index=feature_names)
top20 = importances.sort_values(ascending=False).head(20)
print('=== Top 20 Feature Importances (Random Forest) ===')
display(top20.to_frame('importance').round(4))

=== Top 20 Feature Importances (Random Forest) ===


,importance
carrier_lag_days,0.1658
estimated_delivery_days,0.1330
order_month,0.1132
haversine_distance_km,0.0616
avg_item_freight,0.0458
approval_lag_hrs,0.0427
total_freight,0.0403
total_payment_value,0.0308
total_volume_cm3,0.0304
total_price,0.0299


## Step 7: Save Artifacts

In [10]:
# Save best model
model_path = output_dir / 'best_model.joblib'
joblib.dump(best_rf, model_path)

# Save metrics as JSON
metrics = {
    'model': 'RandomForestClassifier (tuned)',
    'best_params': search.best_params_,
    'validation': {
        'f1_late':  round(rf_f1, 4),
        'roc_auc':  round(rf_roc, 4),
        'pr_auc':   round(rf_pr, 4)
    },
    'test': {
        'f1_late':  round(test_f1, 4),
        'roc_auc':  round(test_roc, 4),
        'pr_auc':   round(test_pr, 4)
    },
    'baseline': {
        'dummy_roc_auc': round(dummy_roc, 4),
        'lr_roc_auc':    round(lr_roc, 4)
    }
}

metrics_path = output_dir / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'✓ Model saved      : {model_path} ({model_path.stat().st_size / (1024*1024):.1f} MB)')
print(f'✓ Metrics saved    : {metrics_path}')
print('\n=======================================================')
print('✓ Notebook 6 (Train, Tune & Evaluate) completed!')
print(f'  Val  ROC-AUC: {rf_roc:.4f}  |  Test ROC-AUC: {test_roc:.4f}')
print(f'  Val  F1(Late): {rf_f1:.4f}  |  Test F1(Late): {test_f1:.4f}')
print('=======================================================')

✓ Model saved      : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/06_train_tune_evaluate/best_model.joblib (12.9 MB)
✓ Metrics saved    : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/06_train_tune_evaluate/metrics.json

✓ Notebook 6 (Train, Tune & Evaluate) completed!
  Val  ROC-AUC: 0.8130  |  Test ROC-AUC: 0.8156
  Val  F1(Late): 0.3977  |  Test F1(Late): 0.3913
